In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))

from matplotlib.ticker import FuncFormatter  # Dependencia de presentación omitida en el original.


# 1.CARGA DE LIBRERIAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colormaps
from collections import defaultdict
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report, confusion_matrix)
from feature_engine.encoding import WoEEncoder
from feature_engine.selection import DropCorrelatedFeatures
import statsmodels.api as sm
import shap
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from feature_engine.imputation import ArbitraryNumberImputer
from feature_engine.discretisation import EqualFrequencyDiscretiser
import time
import xgboost as xgb
import os

# 2.CARGA DE DATOS

In [ ]:
df = pd.read_csv("1.base/base_credit_risk_202510.csv",sep=";")
df.head(5)

In [ ]:
df.shape

# 3.Definición de muestras de desarrollo: resumen mensual 

In [ ]:
resumen = (
    df.groupby('periodo_observacion')
      .agg(
          clientes=('cliente_id', 'nunique'),
          buenos=('indicador_default', lambda x: (x == 0).sum()),
          malos=('indicador_default', lambda x: (x == 1).sum()),
          avg_target=('indicador_default', 'mean')
      )
      .reset_index()
)

# Convertir tasa de default a porcentaje
resumen['avg_target'] = resumen['avg_target'] * 100

# Ordenar por periodo
resumen = resumen.sort_values('periodo_observacion')

resumen

In [ ]:
# ORDENAR LOS PERIODOS
# =================================================
resumen = resumen.sort_values('periodo_observacion').reset_index(drop=True)

# GRAFICO
# =================================================
fig, ax1 = plt.subplots(figsize=(18, 8))
x = np.arange(len(resumen))

# CLIENTES - BARRAS
# =================================================
ax1.bar(
    x,
    resumen['clientes'],
    color='#D6EAF8',
    edgecolor='#5DADE2',
    alpha=0.80,
    label='Clientes'
)

ax1.set_ylabel('Número de clientes', fontsize=12)

ax1.yaxis.set_major_formatter(
    FuncFormatter(lambda valor, pos: f'{valor:,.0f}')
)

# EJE DERECHO - TM
# =================================================
ax2 = ax1.twinx()

ax2.plot(
    x,
    resumen['avg_target'],
    color='black',
    linewidth=2,
    marker='o',
    markersize=6,
    alpha=0.8,
    label='TM'
)

# VALORES DE TM SOBRE LA LÍNEA
# =================================================
for i, valor in enumerate(resumen['avg_target']):

    desplazamiento = 12 if i % 2 == 0 else -15

    ax2.annotate(
        f'{valor:.1f}%',
        xy=(x[i], valor),
        xytext=(0, desplazamiento),
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=9,
        fontweight='bold',
        color='black'
    )

# FORMATO
# =================================================
ax2.grid(linestyle='--', alpha=0.30)

ax1.set_xticks(x)

ax1.set_xticklabels(
    resumen['periodo_observacion'].astype(str),
    rotation=90
)

ax1.set_xlabel('Periodo de observación', fontsize=12)
ax2.set_ylabel('Tasa de default (%)', fontsize=12)

# AJUSTE AUTOMÁTICO DEL EJE DE PORCENTAJES
# =================================================
max_tm = resumen['avg_target'].max()
ax2.set_ylim(0, min(100, max(5, max_tm * 1.30)))

ax2.yaxis.set_major_formatter(
    FuncFormatter(lambda valor, pos: f'{valor:.0f}%')
)

# LEYENDA
# =================================================
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

ax2.legend(
    h1 + h2,
    l1 + l2,
    loc='upper center',
    ncol=2
)

# TITULO
# =================================================
plt.title('Evolución de Clientes y Tasa de Default por Periodo de Observación',fontsize=16,fontweight='bold')

plt.tight_layout()
plt.show()

# 4.Definición de muestras Train / Test / OOT

In [ ]:
meses_oot = [202510,202509, 202508]
oot = df[df['periodo_observacion'].isin(meses_oot)].copy()

In [ ]:
# Ventana de desarrollo
desarrollo = df[(df['periodo_observacion'] <= 202507)].copy()


# 5. División Train / Test

In [ ]:
train, test = train_test_split(
    desarrollo,
    test_size=0.30,
    random_state=123,
    stratify=desarrollo['indicador_default']
)


In [ ]:

for nombre, df_tmp in [
    ('TRAIN', train),
    ('TEST', test),
    ('OOT', oot)
]:
    print(f'\n{nombre}')
    print(f'Registros: {len(df_tmp):,}')
    print(f'Meses: {sorted(df_tmp["periodo_observacion"].unique())}')
    print(df_tmp['indicador_default'].value_counts(normalize=True).round(4))


# 7. Asignación de segmentos

In [ ]:
df['SEGMENTO'] = None

In [ ]:
# Asignación de muestras
df.loc[train.index, 'SEGMENTO'] = 'TRAIN'
df.loc[test.index, 'SEGMENTO'] = 'TEST'
df.loc[oot.index, 'SEGMENTO'] = 'OOT'

In [ ]:
df['SEGMENTO'].value_counts()

In [ ]:
df.head(4)

# 8. GUARDANDO BASE

In [ ]:

os.makedirs("2.OUTPUT", exist_ok=True)

# Guardar dataframe en parquet
df.to_csv("2.OUTPUT/1.1.0.TAB_SEGM_TRAIN_TEST_OOT.csv", index=False)


## Validación de la partición generada
Se compara el archivo recién exportado con hashes y contratos de la salida original; no se lee una partición precalculada como entrada.

In [ ]:
from segmentation_validation import validate_generated_partition
phase1_validation = validate_generated_partition(_repo)
display(pd.DataFrame(phase1_validation['segment_comparison']))